# Car prices — feature engineering
Age, mileage and engine measurements describe different aspects of a vehicle. The features below keep those relationships explicit.

| Feature | Reasoning |
|---|---|
| AgeAtSnapshot | 2016 minus model year; replaces Year |
| LogMileage | Reduces the spread of high mileage values |
| MileagePerYear | Mileage divided by age + 1, avoiding division by zero |
| EngineVolumeMissing | Marks absent or implausible engine measurements |
| Brand, Body, Engine Type, Registration | Represent market categories |

The source has no sale dates or documented mileage units. Age is therefore relative to 2016, and mileage stays in its recorded units. These formulas use predictors only.


## Setup and feature definitions


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML

# Locate data from the notebook folder, project folder or repository root.
locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations
     for candidate in (location, location / "regression" / "car_price_prediction")
     if (candidate / "car_prices_dataset" / "train" / "car_price.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository or car_price_prediction folder.")
DATA_DIR = PROJECT_ROOT / "car_prices_dataset"
TRAIN_PATH = DATA_DIR / "train" / "car_price.csv"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET = "Price"
ID_COLUMN = "RowId"
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [2]:
# Match these split/feature settings to training if you change them.
config = {"random_state": 42, "ground_truth_fraction": 0.2,
          "validation_fraction": 0.2, "reference_year": 2016}
display(pd.Series(config, name="diagnostic_configuration").to_frame())


,diagnostic_configuration
random_state,42.0
ground_truth_fraction,0.2
validation_fraction,0.2
reference_year,2016.0


In [3]:
from sklearn.model_selection import train_test_split

def prepare_data(config):
    """Assign stable source-row IDs, validate labels, deduplicate, then split."""
    raw = pd.read_csv(TRAIN_PATH)
    required = {"Brand", "Price", "Body", "Mileage", "EngineV", "Engine Type", "Registration", "Year", "Model"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Dataset is missing columns: {sorted(missing)}")
    if ID_COLUMN in raw:
        raise ValueError(f"Source column {ID_COLUMN} is reserved for generated row IDs")
    source_columns = raw.columns.tolist()
    raw.insert(0, ID_COLUMN, np.arange(1, len(raw) + 1))
    raw[TARGET] = pd.to_numeric(raw[TARGET], errors="coerce")
    valid_label = np.isfinite(raw[TARGET]) & (raw[TARGET] > 0)
    labeled = raw.loc[valid_label].copy()
    # Ignore generated IDs when identifying identical source records.
    duplicate = labeled.duplicated(subset=source_columns, keep="first")
    usable = labeled.loc[~duplicate].copy()
    if len(usable) < 10 or usable[TARGET].nunique() < 2:
        raise ValueError("Need at least ten labeled unique rows and two distinct positive prices")
    for key in ("ground_truth_fraction", "validation_fraction"):
        if not 0 < config[key] < 0.5:
            raise ValueError(f"{key} must be between 0 and 0.5")
    development, ground = train_test_split(usable, test_size=config["ground_truth_fraction"], random_state=config["random_state"])
    training, validation = train_test_split(development, test_size=config["validation_fraction"], random_state=config["random_state"])
    if config.get("cv_folds", 2) > len(training):
        raise ValueError("cv_folds exceeds training rows")
    assert not (set(training[ID_COLUMN]) & set(validation[ID_COLUMN])
                or set(training[ID_COLUMN]) & set(ground[ID_COLUMN])
                or set(validation[ID_COLUMN]) & set(ground[ID_COLUMN]))
    audit = {"source_rows": len(raw), "excluded_invalid_or_missing_prices": int((~valid_label).sum()),
             "excluded_exact_duplicates": int(duplicate.sum()), "usable_rows": len(usable),
             "training_rows": len(training), "validation_rows": len(validation), "ground_truth_rows": len(ground)}
    return training, validation, ground, audit


In [4]:
def clean_features(X, reference_year):
    """Apply row-wise validity rules; fitted imputers handle remaining gaps."""
    data = X.drop(columns=[ID_COLUMN, TARGET, "Model"], errors="ignore").copy()
    for column in ("Mileage", "EngineV", "Year"):
        data[column] = pd.to_numeric(data[column], errors="coerce")
        data.loc[~np.isfinite(data[column]), column] = np.nan
    data.loc[data["Mileage"] < 0, "Mileage"] = np.nan
    # Treat engine volumes above 6.5 L as unknown for this dataset.
    # Treat them as unknown; retain the car instead of deleting its record.
    data.loc[(data["EngineV"] <= 0) | (data["EngineV"] > 6.5), "EngineV"] = np.nan
    data.loc[(data["Year"] < 1886) | (data["Year"] > reference_year), "Year"] = np.nan
    for column in data.select_dtypes(exclude="number").columns:
        data[column] = data[column].map(lambda value: str(value).strip() if pd.notna(value) else np.nan)
        data[column] = data[column].replace("", np.nan)
    return data

def engineer_features(X, reference_year):
    """Keep domain features deterministic and independent of prices."""
    data = clean_features(X, reference_year)
    data["AgeAtSnapshot"] = reference_year - data["Year"]
    data["LogMileage"] = np.log1p(data["Mileage"])
    # Mileage units are preserved from the source; no unit conversion is assumed.
    data["MileagePerYear"] = data["Mileage"] / (data["AgeAtSnapshot"] + 1)
    data["EngineVolumeMissing"] = data["EngineV"].isna().astype(int)
    # Age and Year encode the same linear information; keep one representation.
    return data.drop(columns="Year")


In [5]:
training, _, _, audit = prepare_data(config)
X = training.drop(columns=TARGET)
engineered = engineer_features(X, config["reference_year"])
new = sorted(set(engineered.columns) - set(X.columns))
display(engineered.head())
display(engineered[new].describe().T)
display(engineered.select_dtypes(include="number").corrwith(np.log1p(training.Price)).sort_values(ascending=False).to_frame("training_log_price_correlation"))
assert ID_COLUMN not in engineered and TARGET not in engineered


,Brand,Body,Mileage,EngineV,Engine Type,Registration,AgeAtSnapshot,LogMileage,MileagePerYear,EngineVolumeMissing
828,Toyota,hatch,92.0,1.3,Petrol,yes,10.0,4.532599,8.363636,0
1250,Volkswagen,crossover,92.0,3.0,Diesel,yes,5.0,4.532599,15.333333,0
1672,Toyota,sedan,165.0,3.5,Gas,yes,8.0,5.111988,18.333333,0
1072,BMW,sedan,246.0,3.0,Diesel,yes,13.0,5.509388,17.571429,0
813,BMW,crossover,82.0,3.0,Petrol,yes,5.0,4.418841,13.666667,0


,count,mean,std,min,25%,50%,75%,max
AgeAtSnapshot,2638.0,9.547005,6.562718,0.0,5.000000,9.000000,13.000000,47.000000
EngineVolumeMissing,2638.0,0.042835,0.202524,0.0,0.000000,0.000000,0.000000,1.000000
LogMileage,2638.0,4.668284,1.344132,0.0,4.510860,5.056246,5.442418,6.846943
MileagePerYear,2638.0,16.158165,9.876274,0.0,11.379721,15.400000,20.000000,200.000000


,training_log_price_correlation
EngineV,0.460929
EngineVolumeMissing,-0.022778
MileagePerYear,-0.065134
LogMileage,-0.527349
Mileage,-0.571262
AgeAtSnapshot,-0.745198


## Redundancy and feature checks
Age replaces Year because the two contain the same information at a fixed snapshot. Raw and log mileage are both included, giving models different ways to represent the mileage relationship. Imputation, scaling and one-hot encoding are fitted within CV. Models fit log1p prices, while saved predictions and scores use price units.

A simple next experiment is to compare the current inputs with a version without the derived mileage features, using the same training folds. Any formula change needs to be applied consistently in notebooks 02–05.


In [6]:
numeric = engineered.select_dtypes(include="number")
px.imshow(numeric.corr(), zmin=-1, zmax=1, color_continuous_scale="RdBu_r", title="Engineered training correlations").show()
for column in ["AgeAtSnapshot", "LogMileage", "MileagePerYear"]:
    px.scatter(engineered.assign(Price=training.Price), x=column, y="Price", title=f"{column} vs training price").show()
